In [3]:
# imports
import os
import numpy as np
import tensorflow as tf
from google.cloud import storage
import tempfile
# import cv2
# from drought_detection.params import BUCKET_NAME, BUCKET_TRAIN_DATA_PATH


sys.path.append("..") # Adds higher directory to python modules path.
from drought_detection.data_handling import parse_visual_rgb, get_img_from_example_rgb


# Load images from GCP and process them

In [8]:
# # set bucket parameters
# BUCKET_NAME = 'wagon-data-batch913-drought_detection'
# BUCKET_TRAIN_DATA_PATH = 'data/train/'

## try to load blobs without for loop (simple way)

In [22]:
# simple way

# GCP bucket parameters
project_name='drought-detection'
bucket_name = 'wagon-data-batch913-drought_detection'

# open storage client
storage_client = storage.Client(project=project_name)

# create blobs from the files starting with "part"
blobs = storage_client.list_blobs(bucket_name, prefix='data/train/part', delimiter='/')

In [23]:
# print blob file names
for blob in blobs:
    print(blob.name)

data/train/part-r-00000
data/train/part-r-00001
data/train/part-r-00002
data/train/part-r-00003
data/train/part-r-00004
data/train/part-r-00005
data/train/part-r-00006
data/train/part-r-00007
data/train/part-r-00008
data/train/part-r-00009
data/train/part-r-00010
data/train/part-r-00011
data/train/part-r-00012
data/train/part-r-00013
data/train/part-r-00014
data/train/part-r-00015
data/train/part-r-00016
data/train/part-r-00017
data/train/part-r-00018
data/train/part-r-00019
data/train/part-r-00020
data/train/part-r-00021
data/train/part-r-00022
data/train/part-r-00023
data/train/part-r-00024
data/train/part-r-00025
data/train/part-r-00026
data/train/part-r-00027
data/train/part-r-00028
data/train/part-r-00029
data/train/part-r-00030
data/train/part-r-00031
data/train/part-r-00032
data/train/part-r-00033
data/train/part-r-00034
data/train/part-r-00035
data/train/part-r-00036
data/train/part-r-00037
data/train/part-r-00038
data/train/part-r-00039
data/train/part-r-00040
data/train/part-

In [26]:
blobs_list = list(blobs)
blobs_list

ValueError: ('Iterator has already started', <google.api_core.page_iterator.HTTPIterator object at 0x18d3d8190>)

In [24]:
print(blobs)

In [25]:
for blob in blobs:
    print(blob)

ValueError: ('Iterator has already started', <google.api_core.page_iterator.HTTPIterator object at 0x18d3d8190>)

In [61]:
images = []


for blob in blobs:
    # transform data format
    parsed_img = parse_visual(blob) # parse satellite image data
    img_sat = get_img_from_example(parsed_img[0]) # convert data to rgbArray with label
    
    # append image to list
    images.append(img_sat)

images

ValueError: ('Iterator has already started', <google.api_core.page_iterator.HTTPIterator object at 0x18f693e80>)

## load blobs in for loop

In [6]:
def get_images_gcp(n=500):
    '''
    This function gets images from the cloud in the correct format.
    The function downloads images into temporary files, does a transformation, and then deletes the temporary file.
    '''
    
    # GCP bucket parameters
    project_name = 'drought-detection'
    bucket_name = 'wagon-data-batch913-drought_detection'

    # open client and get blobs
    storage_client = storage.Client(project=project_name)
    blobs = storage_client.list_blobs(bucket_name, 
                                      prefix='data/train/part', # what do file names start with
                                      delimiter='/', # don't include subdirectories
                                      max_results=n) # max number of blobs to grab
    
    images = []

    for blob in blobs:
        # create temporary file
        _, temp_local_filename = tempfile.mkstemp()

        # Download blob from bucket into temp file
        blob.download_to_filename(temp_local_filename)
        # temp_local_filename = blob.download_as_bytes() # tried downloading TFRecords in different format (as bytes)
        
        # Do stuff to file (transform data format)
        parsed_img = parse_visual_rgb(temp_local_filename) # parse satellite image data
        img_sat = get_img_from_example_rgb(parsed_img[0]) # convert data to rgbArray with label

        # append image (as 3D matrix) to list
        images.append(img_sat)
        
        # remove temporary file
        os.remove(temp_local_filename)
        
    return np.array(images)


In [7]:
# load images
images = get_images_gcp(n=5)

2022-07-13 13:51:52.250619: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.2
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2022-07-13 13:51:52.268782: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:176] None of the MLIR Optimization Passes are enabled (registered 2)


# Check shape of get_images_gcp() output

# format

output is in format: images[satellie image number][0 = 3D image matrix / 1 = label] 

the first index refers to the satellite image number (length depends on how many images you chose to)

In [22]:
# check shape of variable
images.shape # images[sat_img_num][0=3D image matrix/1=label]

(5, 2)

In [8]:
# check first image
images[0] # images[sat_img_num][0=3D image matrix/1=label]

array([array([[[196, 195, 202],
        [196, 201, 202],
        [179, 183, 196],
        ...,
        [133, 154, 176],
        [133, 154, 183],
        [142, 160, 183]],

       [[192, 195, 202],
        [196, 201, 202],
        [192, 195, 202],
        ...,
        [146, 160, 183],
        [142, 160, 183],
        [154, 166, 189]],

       [[188, 189, 202],
        [192, 195, 202],
        [196, 201, 202],
        ...,
        [150, 166, 183],
        [146, 160, 183],
        [154, 166, 189]],

       ...,

       [[158, 160, 176],
        [171, 166, 183],
        [183, 171, 183],
        ...,
        [163, 177, 189],
        [150, 177, 189],
        [163, 177, 196]],

       [[154, 154, 176],
        [167, 160, 183],
        [179, 171, 183],
        ...,
        [154, 171, 189],
        [158, 171, 189],
        [150, 171, 189]],

       [[150, 154, 176],
        [154, 160, 176],
        [167, 166, 183],
        ...,
        [154, 166, 189],
        [158, 166, 189],
        [158, 166

In [21]:
# check shape of first image's data
images[0].shape # images[sat_img_num][0=3D image matrix/1=label]

(2,)

In [13]:
# check shape of first image's size/dimensions (65 pixels x 65 pixels x 3 bands(?))
images[0][0].shape # images[sat_img_num][0=3D image matrix/1=label]

(65, 65, 3)

In [20]:
# check label for first image
images[0][1] # images[sat_img_num][0=3D image matrix/1=label]

0

In [ ]:
images